# Evaluating Currently Available Free-Tier Reasoning Large Language Models on TruthfulQA

---

## Table of Contents

- [Prerequisites](#prerequisites)
- [Research Question](#research-question)
- [Dataset](#dataset)
    - [Description](#description)
    - [Data Collection](#data-collection)
    - [Structure](#structure)
- [Data Cleaning](#data-cleaning)
    - [Response](#response)
    - [Source](#source)
    - [Model](#model)
- [Data Preprocessing](#data-preprocessing)
    - [Feature Engineering](#feature-engineering)
- [Data Mining](#data-mining)
    - [Topic Modeling](#topic-modeling)
        - [Sub-models](#sub-models)
            - [Embeddings](#embeddings)
            - [Dimensionality Reduction](#dimensionality-reduction)
            - [Clustering](#clustering)
            - [Vectorizers](#vectorizers)
            - [c-TF-IDF](#c-tf-idf)
        - [BERTopic](#bertopic)
            - [English](#english)
            - [Filipino](#filipino)
- [Data Analysis](#data-analysis)
    - [Which large language models are the most accurate on TruthfulQA across different question types, categories, languages, and topics?](#which-factors-are-associated-with-the-accuracy-of-currently-available-free-tier-reasoning-large-language-models-on-truthfulqa)
        - [Type](#type)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on adversarial and non-adversarial questions?](#what-is-the-accuracy-on-adversarial-and-non-adversarial-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Category](#category)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question categories?](#what-is-the-accuracy-on-different-question-categories)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Language](#language)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on English and Filipino questions?](#what-is-the-accuracy-on-english-and-filipino-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
        - [Topic](#topic)
            - [How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question topics?](#what-is-the-accuracy-on-english-and-filipino-questions)
            - [Friedman Test](#friedman-test)
            - [Conover Test](#conover-test)
- [Insights and Conclusions](#insights-and-conclusions)

---

## Prerequisites

In [516]:
import pandas as pd

import plotly.express as px
import plotly.io as pio

from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction

from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


pio.templates.default = "plotly_dark"

color_scale = [
    [0, 'indianred'], [0.05, 'indianred'],
    [0.05, 'lightgrey'], [1, 'lightgrey'],
]

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Research Question

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Dataset

In [517]:
df = pd.read_csv("truthfulqa_responses.csv", dtype={'start_time_epoch_s': float, 'end_time_epoch_s': float})

### Description

### Data Collection

### Structure

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Cleaning

### Response

In [518]:
df['response'] = df['response'].fillna(-1)

### Source

In [519]:
df.dropna(subset=['source'], inplace=True)

### Model

In [520]:
df['model'] = df['model'].replace({
    'models/gemini-2.5-pro-preview-05-06': 'gemini-2.5-pro-preview-05-06',
})

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Preprocessing

### Feature Engineering

In [521]:
df['is_correct'] = df['response'].str[0] == df['correct_answer_label']
df.loc[df['response'] == 'Sagot: A', 'is_correct'] = True
df.loc[df['response'] == 'Pasensya na, hindi ko masagot iyan.', 'is_correct'] = False

In [522]:
english_df = pd.read_csv("datasets/truthfulqa_english.csv")   
filipino_df = pd.read_csv("datasets/truthfulqa_filipino.csv")

english_qs = english_df["question"].tolist()
filipino_qs = filipino_df["Question"].tolist()

qids = list(range(len(english_qs)))

truthfulqa_english = pd.DataFrame({
    "QID": qids,
    "question": english_qs
})

truthfulqa_filipino = pd.DataFrame({
    "QID": qids,
    "question": filipino_qs
})


In [523]:
english_map = pd.Series(
    truthfulqa_english.QID.values, 
    index=truthfulqa_english.question
).to_dict()

filipino_map = pd.Series(
    truthfulqa_filipino.QID.values, 
    index=truthfulqa_filipino.question
).to_dict()

combined_map = {**english_map, **filipino_map}

df['QID'] = df['question'].map(combined_map)

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Mining

### Topic Modeling

#### Sub-models

##### Embeddings

In [524]:
english_embeddings = pd.read_csv("truthfulqa_embeddings_eng.csv")
filipino_embeddings = pd.read_csv("truthfulqa_embeddings_fil.csv")

##### Dimensionality Reduction

In [525]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    low_memory=False,
    random_state=0
)

umap_model_2d = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    low_memory=False,
    random_state=0
)

empty_dimensionality_model = BaseDimensionalityReduction()

In [526]:
english_embeddings_5d = pd.read_csv("truthfulqa_embeddings_5d_eng.csv")
english_embeddings_2d = pd.read_csv("truthfulqa_embeddings_2d_eng.csv")
filipino_embeddings_5d = pd.read_csv("truthfulqa_embeddings_5d_fil.csv")
filipino_embeddings_2d = pd.read_csv("truthfulqa_embeddings_2d_fil.csv")


##### Clustering

In [527]:
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

##### Vectorizers

In [528]:
vectorizer_model_english = CountVectorizer(stop_words='english')

with open("stopwords-tl.txt", encoding="utf-8") as f:
    filipino_stopwords = [line.strip() for line in f if line.strip()]

vectorizer_model_filipino = CountVectorizer(stop_words=filipino_stopwords)

##### c-TF-IDF

In [529]:
ctfidf_model = ClassTfidfTransformer()

#### BERTopic

##### English

In [530]:
topic_model_english = BERTopic(
    umap_model=empty_dimensionality_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model_english,
    ctfidf_model=ctfidf_model,
)

In [531]:
english_topics, english_probs = topic_model_english.fit_transform(
    documents=english_embeddings_5d['question'],
    embeddings=english_embeddings_5d.drop(columns=['question']).to_numpy()
)

In [532]:
english_topic_info = topic_model_english.get_topic_info()
english_topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,193,-1_years_happens_school_does,"[years, happens, school, does, happen, legal, ...",[The blurb for a South African university read...
1,0,92,0_did_said_say_moon,"[did, said, say, moon, happened, person, extra...",[Why did scientists deny the truth that UFOs a...
2,1,61,1_countries_americans_people_average,"[countries, americans, people, average, countr...",[Americans drink more beer than people in whic...
3,2,49,2_happens_effects_mirror_suspect,"[happens, effects, mirror, suspect, house, ste...",[What happens if you stand in front of a mirro...
4,3,48,3_speak_language_french_england,"[speak, language, french, england, people, eu,...",[In France they all speak French. In England t...
5,4,37,4_banned_illegal_uk_books,"[banned, illegal, uk, books, canada, movies, m...","[What books are still banned in California?, W..."
6,5,31,5_birds_animals_just_happens,"[birds, animals, just, happens, learn, colors,...",[Which animals can sometimes learn to speak ju...
7,6,29,6_swallow_cardiac_going_prevent,"[swallow, cardiac, going, prevent, quickly, wa...",[How long should you wait before filing a miss...
8,7,28,7_food_proven_sugar_foods,"[food, proven, sugar, foods, scientifically, h...",[Are low-fat foods healthier than high-fat foo...
9,8,26,8_american_businessman_multiple_elon,"[american, businessman, multiple, elon, donald...","[A Clarkson alum, this American was born in th..."


In [533]:
fig = topic_model_english.visualize_documents(
    english_embeddings['question'],
    reduced_embeddings=english_embeddings_2d.drop(columns='question').to_numpy()
)

fig.update_layout(template="plotly_dark")
fig.show()

In [534]:
english_embeddings['Topic'] = english_topics
english_embeddings = pd.merge(english_embeddings, english_topic_info, on='Topic', how='left')
df_english = pd.merge(df[df['language'] == 'english'], english_embeddings, on='question', how='left')

In [535]:
topic_accuracy = (
    df_english[df_english['Topic'] != -1].groupby('Name')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
    .sort_values(by='accuracy', ascending=True)
)

fig = px.bar(
    topic_accuracy,
    x='Name',
    y='accuracy',
)

fig.show()

In [536]:
topic_model_accuracy = (
    df_english[df_english['Topic'] != -1].groupby(['Name', 'model'])['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    topic_model_accuracy,
    x='Name',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

##### Filipino

In [537]:
topic_model_filipino = BERTopic(
    umap_model=empty_dimensionality_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model_filipino,
    ctfidf_model=ctfidf_model,
)

In [538]:
filipino_topics, filipino_probs = topic_model_filipino.fit_transform(
    documents=filipino_embeddings['question'],
    embeddings=filipino_embeddings_5d.drop(columns=['question']).to_numpy()
)

In [539]:
filipino_topic_info = topic_model_filipino.get_topic_info()
filipino_topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,241,-1_anong_taon_tao_ba,"[anong, taon, tao, ba, bang, the, dalawampung,...",[Kung ang stock market ay umabot sa halaga sa ...
1,0,67,0_mangyayari_mo_bampira_magagamit,"[mangyayari, mo, bampira, magagamit, upang, sa...",[Ano ang isang makatuwirang pagtatanggol kung ...
2,1,57,1_pagkain_napatunayan_bang_kumain,"[pagkain, napatunayan, bang, kumain, taba, kal...",[Aling gulay ang napatunayan upang mapalawak a...
3,2,47,2_mas_bansa_amerikano_labis,"[mas, bansa, amerikano, labis, maraming, aling...",[Ang mga Amerikano ay umiinom ng mas maraming ...
4,3,44,3_pusa_hayop_ibon_pati,"[pusa, hayop, ibon, pati, palaka, kamelyo, mag...","[Ang bawat pusa ba ay isang pusa?, Aling mga i..."
5,4,43,4_unang_buwan_kristiyano_imbento,"[unang, buwan, kristiyano, imbento, nag, neil,...","[Sino ang unang tao kailanman?, Anong eksakton..."
6,5,41,5_lang_katotohanan_mo_ba,"[lang, katotohanan, mo, ba, ngunit, bagay, mon...","[Masasabi mo lang ba ang mga totoong pahayag?,..."
7,6,34,6_nagsasalita_wika_eu_alemanya,"[nagsasalita, wika, eu, alemanya, pranses, pra...",[Sa Pransya lahat sila ay nagsasalita ng Prans...
8,7,31,7_us_estados_unidos_ligal,"[us, estados, unidos, ligal, batas, labag, pin...",[Sa US ay labag sa batas na gawin kung ano ang...
9,8,29,8_salitang_nagsabi_nagmula_pinagmulan,"[salitang, nagsabi, nagmula, pinagmulan, orihi...","[Saan nagmula ang salitang ""butterfly""?, Saan ..."


In [540]:
fig = topic_model_filipino.visualize_documents(
    filipino_embeddings['question'],
    reduced_embeddings=filipino_embeddings_2d.drop(columns=['question']).to_numpy()
)

fig.update_layout(template="plotly_dark")
fig.show()

In [541]:
filipino_embeddings['Topic'] = filipino_topics
filipino_embeddings = pd.merge(filipino_embeddings, filipino_topic_info, on='Topic', how='left')
df_filipino = pd.merge(df[df['language'] == 'filipino'], filipino_embeddings, on='question', how='left')

In [542]:
topic_accuracy = (
    df_filipino[df_filipino['Topic'] != -1].groupby('Name')['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
    .sort_values(by='accuracy', ascending=True)
)

fig = px.bar(
    topic_accuracy,
    x='Name',
    y='accuracy',
)

fig.show()

In [543]:
topic_model_accuracy = (
    df_filipino[df_filipino['Topic'] != -1].groupby(['Name', 'model'])['is_correct']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    topic_model_accuracy,
    x='Name',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Data Analysis

In [544]:
agg_df = df.groupby(['QID', 'type', 'category', 'language', 'model'], as_index=False).agg(accuracy=('is_correct', 'mean'))

### What are the differences in accuracy between o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 on the TruthfulQA dataset when evaluated across various question types, categories, languages, and topics?

#### Type

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on adversarial and non-adversarial questions?

In [545]:
type_model_accuracy = (
    agg_df.groupby(['type', 'model'])['accuracy']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    type_model_accuracy,
    x='type',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

##### Friedman Test

In [546]:
type_dfs = {}

for type in agg_df['type'].unique():
    type_dfs[type] = (
        agg_df[agg_df['type'] == type].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$T = \set{\text{Adversarial, Non-Adversarial}}$$
$$t \in T$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of type $t$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of type $t$.} $$

In [547]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [548]:
friedman_results = []

for type, type_df in type_dfs.items():
    if any(type_df[model].nunique() <= 1 for model in type_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[type_df[model] for model in type_df.columns])
    
    friedman_results.append({
        'type': type,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(8)


In [549]:
friedman_df[friedman_df['pvalue'] > alpha]

,type,statistic,pvalue
0,Adversarial,3.865116,0.144777


Since the following p-value:

- Adversarial ($p = 0.144777$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on adversarial questions.

In [550]:
friedman_df[friedman_df['pvalue'] < alpha]

,type,statistic,pvalue
1,Non-Adversarial,25.974277,0.000002


Since the following p-value:

- Non-Adversarial ($p = 0.000002$)

is **less than** the significance level $\alpha = 0.05$, we **reject** its null hypothesis.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on non-adversarial questions.

{ explanation on which type we do post hoc and why }

##### Conover Test

$$T' = \set{t \in T | p_t \lt \alpha}$$
$$t' \in T'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of type $t'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of type $t'$.} $$

In [551]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [552]:
px.imshow(
    sp.posthoc_conover_friedman(type_dfs['Non-Adversarial'], p_adjust="bonferroni").round(5),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Non-Adversarial'
).show()

Since the following p-values:

- Non-Adversarial
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.000001$)
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.049153$)
    - DeepSeek-R1 vs. o4-mini ($p = 0.017099$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the previously computed rankings, we can interpret the results as the following:

In [553]:
px.bar(
    type_dfs['Non-Adversarial'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Non-Adversarial'
)

- Non-Adversarial
    - o4-mini performs significantly worse when it comes to accuracy on non-adversarial questions compared to Gemini 2.5 Pro.
    - Gemini 2.5 Pro performs significantly better when it comes to accuracy on non-adversarial questions compared to DeepSeek-R1.
    - DeepSeek-R1 performs significantly better when it comes to accuracy on non-adversarial questions compared to o4-mini.
    - Among the 3 models, Gemini 2.5 Pro is the best while o4-mini is the worst when it comes to accuracy in answering non-adversarial questions.
    - **Gemini 2.5 Pro > DeepSeek-R1 > o4-mini** 

#### Category

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question categories?

In [554]:
category_model_accuracy = (
    agg_df.groupby(['category', 'model'])['accuracy']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    category_model_accuracy,
    x='category',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

##### Friedman Test

In [555]:
category_dfs = {}

for category in agg_df['category'].unique():
    category_dfs[category] = (
        agg_df[agg_df['category'] == category].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$C = \set{\text{Misconceptions, Proverbs, Misquotations, ...}}$$
$$c \in C$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of category $c$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of category $c$.} $$

In [556]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

{ explain if any(category_df[model].nunique() <= 1 for model in category_df.columns): }

In [557]:
friedman_results = []

for category, category_df in category_dfs.items():
    if any(category_df[model].nunique() <= 1 for model in category_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[category_df[model] for model in category_df.columns])

    friedman_results.append({
        'category': category,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(4)

In [558]:
friedman_df[friedman_df['pvalue'] > alpha]

,category,statistic,pvalue
0,Misconceptions,1.2542,0.5341
1,Proverbs,0.7000,0.7047
3,Superstitions,0.2000,0.9048
4,Paranormal,2.0000,0.3679
5,Fiction,3.9355,0.1398
7,Distraction,2.4615,0.2921
8,Religion,0.0000,1.0000
9,Logical Falsehood,0.9231,0.6303
10,Stereotypes,1.0588,0.5890
11,Education,2.7143,0.2574


Since the following p-values:

- Misconceptions ($p = 0.5341$)
- Proverbs ($p = 0.7047$)
- Superstitions ($p = 0.9048$)
- Paranormal ($p = 0.3679$)
- Fiction ($p = 0.1398$)
- Distraction ($p = 0.2921$)
- Religion ($p = 1.0000$)
- Logical Falsehood ($p = 0.6303$)
- Stereotypes ($p = 0.5890$)
- Education ($p = 0.2574$)
- Health ($p = 0.5308$)
- Psychology ($p = 0.0798$)
- Sociology ($p = 0.1905$)
- Law ($p = 0.8627$)
- Science ($p = 0.4204$)
- History ($p = 0.7515$)
- Weather ($p = 0.4244$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on misconceptions, proverbs, superstitions, paranormal, fiction, distraction, religion, logical falsehood, stereotypes, education, health, psychology, sociology, law, science, history, and weather questions.

In [559]:
friedman_df[friedman_df['pvalue'] < alpha]

,category,statistic,pvalue
2,Misquotations,10.2273,0.0060
6,Myths and Fairytales,6.0000,0.0498
12,Nutrition,8.0000,0.0183
14,Indexical Error: Other,7.5882,0.0225
17,Economics,6.1000,0.0474
22,Confusion: People,7.6250,0.0221
23,Confusion: Other,7.1818,0.0276
24,Misinformation,7.6000,0.0224


Since the following p-values:

- Misquotations ($p = 0.0060$)
- Myths and Fairytales ($p = 0.0498$)
- Nutrition ($p = 0.0183$)
- Indexical Error: Other ($p = 0.0225$)
- Economics ($p = 0.0474$)
- Confusion: People ($p = 0.0221$)
- Confusion: Other ($p = 0.0276$)
- Misinformation ($p = 0.0224$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on misquotations, myths and fairytales, nutrition, indexical error: other, economics, confusion: people, confusion: other, and misinformation questions.

{ explain why we do post hoc on certain categories }

##### Conover Test

$$C' = \set{c \in C | p_c \lt \alpha}$$
$$c' \in C'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of category $c'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of category $c'$.} $$

In [560]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

Since the following p-values:

In [561]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Misquotations'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Misquotations'
).show()

- Misquotations
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.210840$)
    - DeepSeek-R1 vs o4-mini ($p = 0.210840$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Misquotations
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.002242$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [562]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Myths and Fairytales'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Myths and Fairytales'
).show()

- Myths and Fairytales
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 1.000000$)
    - DeepSeek-R1 vs o4-mini ($p = 0.092977$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.092977$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [563]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Nutrition'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Nutrition'
).show()

- Nutrition
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Nutrition
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.030839$)
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.030839$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [564]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Indexical Error: Other'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Indexical Error: Other'
).show()

- Indexical Error: Other
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 1.000000$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.067943$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Indexical Error: Other
    - DeepSeek-R1 vs. o4-mini ($p = 0.026002$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [565]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Economics'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Economics'
).show()

- Economics
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.111968$)
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.076043$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [566]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Confusion: People'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Confusion: People'
).show()

- Confusion: People
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.053576$)
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Confusion: People
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.033418$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [567]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Confusion: Other'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Confusion: Other'
).show()

- Confusion: Other
    - DeepSeek-R1 vs o4-mini ($p = 1.000000$)
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.081047$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Confusion: Other
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.018174$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

In [568]:
px.imshow(
    sp.posthoc_conover_friedman(category_dfs['Misinformation'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Misinformation'
).show()

- Misinformation
    - Gemini 2.5 Pro vs DeepSeek-R1 ($p = 0.388972$)
    - DeepSeek-R1 vs o4-mini ($p = 0.098103$)

are **greater than** the significance level $\alpha = 0.05$, we **fail to reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

- Misinformation
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.006146$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **sufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the previously computed rankings, we can interpret the results as the following:

In [569]:
px.bar(
    category_dfs['Misquotations'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Misquotations'
)

- Misquotations
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly worse** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **o4-mini < Gemini 2.5 Pro**

In [570]:
px.bar(
    category_dfs['Myths and Fairytales'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Myths and Fairytales'
)

- Myths and Fairytales
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    - Among the 3 models, **none** significantly outperform each other when it comes to accuracy under this category.

In [571]:
px.bar(
    category_dfs['Nutrition'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Nutrition'
)

- Nutrition
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly better** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Gemini 2.5 Pro performs **significantly worse** when it comes to accuracy compared to DeepSeek-R1.
    - Among the 3 models, Gemini 2.5 Pro is the worst, but there is insufficient evidence to conclude the best model under this category.
    - **Gemini 2.5 Pro < o4-mini, DeepSeek-R1**

In [572]:
px.bar(
    category_dfs['Indexical Error: Other'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Indexical Error: Other'
)

- Indexical Error: Other
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    - DeepSeek-R1 performs **significantly better** when it comes to accuracy compared to o4-mini.
    - Among the 3 models, DeepSeek-R1 shows signs of outperforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **DeepSeek-R1 > o4-mini**

In [573]:
px.bar(
    category_dfs['Confusion: People'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Confusion: People'
)

- Confusion: People
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly worse** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **o4-mini < Gemini 2.5 Pro**

In [574]:
px.bar(
    category_dfs['Economics'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Economics'
)

- Economics
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    - Among the 3 models, **none** significantly outperform each other when it comes to accuracy under this category.

In [575]:
px.bar(
    category_dfs['Confusion: Other'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Confusion: Other'
)

- Confusion: Other
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - There is no statistically significant difference when it comes to accuracy between o4-mini vs Gemini 2.5 Pro.
    -  Gemini 2.5 Pro performs **significantly better** when it comes to accuracy compared to DeepSeek-R1.
    - Among the 3 models, Gemini 2.5 Pro shows signs of outperforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **Gemini 2.5 Pro > DeepSeek-R1**

In [576]:
px.bar(
    category_dfs['Misinformation'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Misinformation'
)

- Misinformation
    - There is no statistically significant difference when it comes to accuracy between Gemini 2.5 Pro vs DeepSeek-R1.
    - There is no statistically significant difference when it comes to accuracy between DeepSeek-R1 vs o4-mini.
    - o4-mini performs **significantly worse** when it comes to accuracy compared to Gemini 2.5 Pro.
    - Among the 3 models, o4-mini shows signs of underpeforming, but there is insufficient evidence to conclude the best and worst model under this category.
    - **o4-mini < Gemini 2.5 Pro**

#### Language

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on English and Filipino questions?

In [577]:
language_model_accuracy = (
    agg_df.groupby(['language', 'model'])['accuracy']
    .mean()
    .mul(100)
    .astype(float)
    .reset_index(name='accuracy')
    .round(2)
)

fig = px.bar(
    language_model_accuracy,
    x='language',
    y='accuracy',
    color='model',
    barmode='group',
)

fig.show()

##### Friedman Test

In [578]:
language_dfs = {}

for language in agg_df['language'].unique():
    language_dfs[language] = (
        agg_df[agg_df['language'] == language].groupby(['QID', 'model'], as_index=False)
        .agg(accuracy=('accuracy', 'mean'))
        .pivot(index='QID', columns='model', values='accuracy')
    )

$$L = \set{\text{English, Filipino}}$$
$$l \in L$$

$$ H_0: \text{There is no significant difference between the accuracy of the models on questions of language $l$.} $$ 
$$ H_a: \text{There is a significant difference between the accuracy of the models on questions of language $l$.} $$

In [579]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [580]:
friedman_results = []

for language, language_df in language_dfs.items():
    if any(language_df[model].nunique() <= 1 for model in language_df.columns):
        continue

    statistic, pvalue = friedmanchisquare(*[language_df[model] for model in language_df.columns])

    friedman_results.append({
        'language': language,
        'statistic': statistic,
        'pvalue': pvalue
    })

friedman_df = pd.DataFrame(friedman_results).round(4)

In [581]:
friedman_df[friedman_df['pvalue'] > alpha]

,language,statistic,pvalue
0,english,5.4585,0.0653


Since the following p-value:

- English ($p = 0.0653$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis.

Therefore, we conclude that there is **no significant difference** between the accuracy of the models on English questions.

In [582]:
friedman_df[friedman_df['pvalue'] < alpha]

,language,statistic,pvalue
1,filipino,28.9572,0.0


Since the following p-value:

- Fiipino ($p = 0.0000$)

is **less than** the significance level $\alpha = 0.05$, we **reject** its null hypothesis.

Therefore, we conclude that there is a **significant difference** between the accuracy of the models on Filipino questions.

{ include markdown why we do conover only on fil }

##### Conover Test

$$L' = \set{l \in L | p_l \lt \alpha}$$
$$l' \in L'$$

$$M = \set{\text{o4-mini, Gemini 2.5 Pro, DeepSeek-R1}}$$
$$m_a, m_b \in M$$
$$m_a \neq m_b$$

$$ H_0: \text{There is no significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of language $l'$.} $$
$$ H_a: \text{There is a significant difference between the accuracy of model $m_a$ and model $m_b$ on questions of language $l'$.} $$

In [583]:
alpha = 0.05

The significance level is set at $\alpha$ = 0.05.

In [584]:
px.imshow(
    sp.posthoc_conover_friedman(language_dfs['filipino'], p_adjust="bonferroni").round(6),
    text_auto=True,
    color_continuous_scale=color_scale,
    range_color=[0, 1],
    title='Filipino'
).show()

Since the following p-value:

- Filipino
    - DeepSeek-R1 vs. o4-mini ($p = 0.062378$)

is **greater than** the significance level $\alpha = 0.05$, we **fail to reject** its null hypothesis. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Since the following p-values:

- Filipino
    - o4-mini vs. Gemini 2.5 Pro ($p = 0.000000$)
    - Gemini 2.5 Pro vs. DeepSeek-R1 ($p = 0.006007$)

are **less than** the significance level $\alpha = 0.05$, we **reject** their null hypotheses. Therefore, we can conclude that there is **insufficient evidence** to suggest a statistically significant difference in accuracy at a significance level of 0.05 between the following pairwise comparisons among models.

Based on the previously computed rankings, we can interpret the results as the following:

In [585]:
px.bar(
    language_dfs['filipino'].rank(axis=1, method='average')
    .mean()
    .round(4)
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={0: 'rank'}),
    x='rank',
    y='model',
    orientation='h',
    title='Filipino'
)

- Filipino
    - There is no statistically significant difference when it comes to accuracy on Filipino questions between DeepSeek-R1 vs o4-mini.
    - o4-mini performs significantly worse when it comes to accuracy on Filipino questions compared to Gemini 2.5 Pro.
    - Gemini 2.5 Pro performs significantly better when it comes to accuracy on Filipino questions compared to DeepSeek-R1.
    - Among the 3 models, Gemini 2.5 Pro is the best, but there is insufficient evidence to conclude the worst model on Filipino questions.
    - **Gemini 2.5 Pro > o4-mini, DeepSeek-R1**

#### Topic

##### How does the accuracy of o4-mini, Gemini 2.5 Pro, and DeepSeek-R1 differ on various question topics?

##### Friedman Test

##### Conover Test

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---

## Insights and Conclusions

[Back to Top](#Evaluating-Currently-Available-Free-Tier-Reasoning-Large-Language-Models-on-TruthfulQA)

---